In [2]:
import numpy as np
import pandas as pd
from functools import partial
from enum import Enum, auto
import json


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
class State(str, Enum):
    OREGON = "oregon"
    SOUTH_CAROLINA = "south_carolina"
class Race(str, Enum):
    WHITE = "white"
    HISPANIC = "hispanic"
    BLACK = "black"

In [15]:
# Configs
recency_election_scores = 1
minority_effectiveness_score_threshold = .6
in_group_calibration = .7 # The penalty for the preferred candidate not being in_group, always 1 when they are

state = State.SOUTH_CAROLINA
races= [Race.BLACK.value, Race.WHITE.value, Race.HISPANIC.value]

recency = 1
in_group_mapping = {Race.WHITE.value: in_group_calibration, Race.HISPANIC.value: in_group_calibration, Race.BLACK.value: 1}


logit_a = 9
logit_b = logit_a / -2


In [16]:




# MAIN_PATH = f"/content/drive/MyDrive/U4/Spring/CSE416/Seawulf/On Seawulf/{state}"
VAP_PATH = f'/content/drive/MyDrive/U4/Spring/CSE416/Preprocessing/VAP2/my_preprocessing1_out/sw_{state.value}'
PATH_D = f"{VAP_PATH}/data"
PATH_G = f"{VAP_PATH}/graphs"
state_data_df = pd.read_csv(f"{PATH_D}/combined_data.csv")
num_districts= len(state_data_df['district'].unique())
POP_THRESHOLD = None
if any(state_data_df[race].sum() > 400000 for race in races if race.lower() != Race.WHITE.value):
    POP_THRESHOLD = 400000
else:
    POP_THRESHOLD = 200000
minority_sig_groups = [race for race in races if (sum(state_data_df[race])> POP_THRESHOLD and race.lower() != Race.WHITE.value) ]

ei_conf_df = pd.read_csv(f"{PATH_D}/ei_statewide.csv")
ei_conf_all_races = dict(zip(ei_conf_df['racial_group'], ei_conf_df['confidence']))


class Group_Control(Enum):
    DEFAULT = auto()    # Value: 1
    MIN_CAND_VOTE_SHARE = auto()  # Value: 2

group_control = Group_Control.DEFAULT


state_extra_details_json = {}
with open(f"{PATH_D}/state_details.json", "r") as f:
    state_extra_details_json = json.load(f)


In [ ]:
def prob_conf_conversion(distr_score):

  calibrated_score = 1/(1+np.exp(-(logit_a * distr_score + logit_b)))
  return calibrated_score

def collect_default_group_control(district_id, df,race):
    district_dict = {}
    district_dict[f'{race}_pop'] = df[df['district_id'] ==district_id][race] .sum()
    district_dict['total_pop'] = df[df['district_id'] ==district_id]['total'].sum()
    return district_dict
def evaluate_default_group_control(district_dict, race):
  group_control = min(1,district_dict[f'{race}_pop']/district_dict['total_pop'] * 2)
  return group_control

# def collect_min_cand_group_control(district_id, df,race):
#     district_dict = {}
#     district_dict['democratic_votes'] = df['democratic_votes']
#     pass

def calculate_effectiveness_score(df,ei,in_group, race,group_control_collect_function,
                                  group_control_evaluate_function):
  district_eff_scores= {}
  for district in (range(num_districts)):
    district = district +1

    district_dict = group_control_collect_function(district,df,race)
    district_score = recency * ei[race] * in_group[race]
    group_control = group_control_evaluate_function(district_dict, race)
    district_score *= group_control

    district_calibrated_score = prob_conf_conversion(district_score)
    district_eff_scores[district] = district_calibrated_score


  return district_eff_scores
def calculate_raw_eff_score(df,ei,in_group, race,group_control_collect_function,
                                  group_control_evaluate_function):
  district_eff_scores= {}
  for district in (range(num_districts)):
    district = district +1

    district_dict = group_control_collect_function(district,df,race)
    district_score = recency * ei[race] * in_group[race]
    group_control = group_control_evaluate_function(district_dict, race)
    district_score *= group_control
    district_eff_scores[district] = district_score

  return district_eff_scores
def calculate_raw_effectiveness_score(df,ei,in_group, race):
  return calculate_raw_eff_score(df,ei,in_group, race,
                 collect_default_group_control,
                 evaluate_default_group_control)

def calculate_default_eff_score(df,ei,in_group,race):
  return calculate_effectiveness_score(df,ei,in_group, race,
                 collect_default_group_control,
                 evaluate_default_group_control)
# def calculate_minority_candidate_eff_score(distr_dict, df,race):
#   return calculate_effectiveness_score(df,race,
#   )
write_out_path = f"{PATH_D}/state_details.json"
try:
    with open(write_out_path, "r") as f:
        state_extra_details_json = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    # Fallback if file doesn't exist or is empty
    state_extra_details_json = {}
if "effective_district_score" not in state_extra_details_json:
    state_extra_details_json["effective_district_score"] = {}
if "effective_district_count" not in state_extra_details_json:
    state_extra_details_json["effective_district_count"] = {}
if "effective_district_score_raw" not in state_extra_details_json:
    state_extra_details_json["effective_district_score_raw"] = {}
from pprint import pprint
for race in minority_sig_groups:
    calibrated_scores = calculate_default_eff_score(state_data_df, ei_conf_all_races, in_group_mapping, race)
    raw_scores = calculate_raw_effectiveness_score(state_data_df, ei_conf_all_races, in_group_mapping, race)

    clean_scores = {str(dist): float(score) for dist, score in calibrated_scores.items()}
    clean_raw_scores = {str(dist): float(score) for dist, score in raw_scores.items()}


    effective_count = sum(1 for score in clean_scores.values() if score >= minority_effectiveness_score_threshold)

    state_extra_details_json["effective_district_score"][race] = clean_scores
    state_extra_details_json["effective_district_score_raw"][race] = clean_raw_scores
    state_extra_details_json["effective_district_count"][race] = effective_count


# 4. Write back into the original JSON file
with open(write_out_path, "w") as f:
    json.dump(state_extra_details_json, f, indent=4)

print(f"Updated JSON successfully for races: {', '.join(minority_sig_groups)} in {state.value}")

Updated JSON successfully for races: hispanic in oregon


In [ ]:
def gui_3_out(state_input_data, output_file):
  groupRoughProportionality = []
  total_population = state_data_df['total'].sum()
  state_cumulated = None
  with open(state_input_data, 'r') as f:
    state_cumulated = json.load(f)
  for minority in minority_sig_groups:
    population= state_data_df[minority].sum()
    pop_percentage = float(f'{population/total_population:.2f}')
    effective_district_count = int(state_cumulated["effective_district_count"][minority])
    output_json = {'groupKey': f'{minority.upper()}', 'label': f'{minority.lower()}',
                   'enactedEffectiveDistricts': f'{effective_district_count}',
                   'cvapShare' : f'{pop_percentage}',
                   'roughProportionalityRatio': f'{(effective_district_count / pop_percentage) : .2f}'}
    groupRoughProportionality.append(output_json)
  with open(output_file, 'w') as f:
    json.dump(groupRoughProportionality, f, indent=4)

gui_3_out('6sc.json', 'gui6sc.json')


In [19]:
'''
{
  "schemaVersion": "v1",
  "state": "OR",
  "election": "2024_pres",
  "districts": [
    {
      "districtNumber": 1,
      "representative": "Suzanne Bonamici",
      "party": "Democratic",
      "racialEthnicGroup": "White",
      "voteMargin2024": 37.6,
      "effectivenessScore": 0.41,
      "calibratedEffectivenessScore": 0.38
    },
    {
      "districtNumber": 2,
      "representative": "Cliff Bentz",
      "party": "Republican",
      "racialEthnicGroup": "White",
      "voteMargin2024": -27.1,
      "effectivenessScore": 0.29,
      "calibratedEffectivenessScore": 0.27
    }
  ]
}
'''

def gui_6_out(input_json, corrected_json_out):
  district_jsons = []
  input_data = None
  with open(input_json, 'r') as f:
    input_data = json.load(f)
  current_raw_eff_scores = state_extra_details_json['effective_district_score']['black']
  current_cal_eff_scores = state_extra_details_json['effective_district_score_raw']['black']
  # if(STATE == State.OREGON.value):
  #   current_raw_eff_scores = []
  #   current_cal_eff_scores = []
  # else:
  #   current_raw_eff_scores = []
  #   current_cal_eff_scores = []

  prev_district_jsons = input_data['districts']
  print(current_raw_eff_scores)
  for index, district_data in enumerate(prev_district_jsons):
    district_data['effectivenessScore'] = current_raw_eff_scores[f'{index+1}']
    district_data['calibratedEffectivenessScore'] = current_cal_eff_scores[f'{index+1}']
  with open(corrected_json_out, 'w') as f:
    json.dump(input_data, f, indent=4)


gui_6_out('6sc.json', '6SC.json')


{'1': 0.17173884667813002, '2': 0.4484278182435083, '3': 0.18092763760705585, '4': 0.21564071933224202, '5': 0.4396081849541272, '6': 0.973128471356273, '7': 0.4680421126562885}


In [ ]:
import json
import numpy as np
import os


def prob_conf_conversion(distr_score, logit_a=logit_a, logit_b=logit_b):
    # Calculate the calibrated score using a logistic function
    calibrated_score = 1 / (1 + np.exp(-(logit_a * distr_score + logit_b)))
    return calibrated_score

def is_effective(total, min_pop, in_group):
    score_group = 1.0 if in_group else 0.7
    group_scored = 0.997 * score_group * min(1.0, 2 * min_pop / total)
    cal_score = prob_conf_conversion(group_scored)
    return cal_score >= 0.6

topojson_file = 'vra_electoral_cracking_min.topojson'
'''
or:
vra_cracking_districts_min_hispanic[4]
vra_electoral_cracking_min

'''
# (Make sure your prob_conf_conversion and is_effective functions are defined above this)

folder_path = '/content/drive/MyDrive/U4/Spring/CSE416/Preprocessing/VAP2/my_preprocessing1_out/output/south_carolina/topojson/'

# Set the target minority and check membership logically
minority = 'black'
in_group = (minority == 'hispanic')

# Iterate through every file in the target folder
for topojson_file in os.listdir(folder_path):
    # Skip any files that aren't topojson
    if not topojson_file.endswith('.topojson'):
        continue

    topojson_path = os.path.join(folder_path, topojson_file)

    # Safely open and parse the TopoJSON
    try:
        with open(topojson_path) as f:
            topo_data = json.load(f)

        district_datas = topo_data['objects']['data']['geometries']

        # Optional: Print a header for each file to make the console easier to read
        print(f"\n--- Results for {topojson_file} ---")

        for district_data in district_datas:
            props = district_data['properties']

            total = props['total']
            minority_pop = props[minority]
            district_id = props['district_id']

            # Check effectiveness
            effective = is_effective(total, minority_pop, in_group)

            if(effective):
              # Print the file name next to the output
              # print(f"[{topojson_file}] district_id {district_id} is effective {effective}")
              print(district_id)

    except Exception as e:
        print(f"Error processing {topojson_file}: {e}")

    '''
OR:
--- Results for vra_cracking_districts_min_hispanic.topojson ---
4

--- Results for vra_packing_districts_max_hispanic.topojson ---
4

--- Results for vra_electoral_cracking_min.topojson ---

--- Results for vra_electoral_packing_max.topojson ---

--- Results for vra_majority_minority_district_count_max_hispanic.topojson ---
5


--- Results for vra_majority_minority_district_count_min_hispanic.topojson ---
5

--- Results for race_blind_packing_districts_max_hispanic.topojson ---
5


--- Results for race_blind_cracking_districts_min_hispanic.topojson ---
5


--- Results for race_blind_electoral_cracking_min.topojson ---

--- Results for race_blind_electoral_packing_max.topojson ---

--- Results for race_blind_majority_minority_district_count_max_hispanic.topojson ---
5


--- Results for race_blind_majority_minority_district_count_min_hispanic.topojson ---
5



SC:

--- Results for vra_packing_districts_max_black.topojson ---

--- Results for vra_cracking_districts_min_black.topojson ---
3
4

--- Results for vra_electoral_packing_max.topojson ---

--- Results for vra_electoral_cracking_min.topojson ---

--- Results for vra_majority_minority_district_count_max_black.topojson ---
6

--- Results for vra_majority_minority_district_count_min_black.topojson ---

--- Results for race_blind_packing_districts_max_black.topojson ---
6

--- Results for race_blind_cracking_districts_min_black.topojson ---
6

--- Results for race_blind_electoral_packing_max.topojson ---

--- Results for race_blind_electoral_cracking_min.topojson ---

--- Results for race_blind_majority_minority_district_count_max_black.topojson ---
6

--- Results for race_blind_majority_minority_district_count_min_black.topojson ---
6
    '''


--- Results for vra_packing_districts_max_black.topojson ---

--- Results for vra_cracking_districts_min_black.topojson ---
3
4

--- Results for vra_electoral_packing_max.topojson ---

--- Results for vra_electoral_cracking_min.topojson ---

--- Results for vra_majority_minority_district_count_max_black.topojson ---
6

--- Results for vra_majority_minority_district_count_min_black.topojson ---

--- Results for race_blind_packing_districts_max_black.topojson ---
6

--- Results for race_blind_cracking_districts_min_black.topojson ---
6

--- Results for race_blind_electoral_packing_max.topojson ---

--- Results for race_blind_electoral_cracking_min.topojson ---

--- Results for race_blind_majority_minority_district_count_max_black.topojson ---
6

--- Results for race_blind_majority_minority_district_count_min_black.topojson ---
6


In [ ]:
path = '/content/drive/MyDrive/U4/Spring/CSE416/Preprocessing/VAP2/my_preprocessing1_out/output/sw_out1/regular_bw.json'
with open(path, 'r') as f:
  data = json.load(f)
update_path = 'rbw.json'
with open(update_path, 'r') as f:
  update_json = json.load(f)

update_json['south_carolina']['vra'] = data['south_carolina']['vra']
with open('regularbw.json', 'w') as f:
  json.dump(update_json, f, indent=4)
print(data['south_carolina']['vra'])

JSONDecodeError: Expecting value: line 171622 column 17 (char 4194304)